# Correctness validation of `monaqa2.qiskit2`

Tests use exact matrix comparisons, including phases, and classical evaluation of reversible arithmetic. All matrices use Qiskit's little-endian convention: qubit 0 is the least-significant bit, and tensor factors are ordered from the highest qubit down to qubit 0. No dense Boltzmann-coin matrix is constructed.


## Import


In [1]:
import sys
from itertools import product
from functools import reduce
import numpy as np
from scipy.linalg import expm
from qiskit.circuit import QuantumCircuit, Gate
from qiskit.quantum_info import Operator, Statevector

if ".." not in sys.path:
    sys.path.append("..")

from monaqa2.mcmc import transition
from monaqa2.mcmc.model import IsingModel, RandomSherringtonKirkpatrickModel
from monaqa2.qiskit2 import (
    Ccx, Rx, Ry, Crx, Cry, Crz, Ccrx, Ccry, Ccrz, Rzz, Crzz, Ccrzz,
    GivensRotation, ControlledGivensRotation, ReversibleFanout, ReversibleZeroFlag,
    ProposalLocal, ProposalUniform, ProposalHamSim, ProposalHamSimRandomized,
    ReversibleCarryLookaheadAdder, ReversibleWallaceAdder, ReversibleConditionalLoader,
    ReversibleDeltaEnergy, ReversiblePositivePart, ReversibleComparator,
    RotationalAcceptance, AcceptPath, Reflection, Walk,
)

ATOL = 1e-10
MAX_UNITARY_QUBITS = 10  # Dense complex matrices are already 16 MiB at this limit.
MAX_STATEVECTOR_QUBITS = 14
rng = np.random.default_rng(214)

## Linear algebra utilities

`IsingModel(J=..., h=...)` requires a one-dimensional NumPy field vector and a symmetric, zero-diagonal coupling matrix. Instances below use `RandomSherringtonKirkpatrickModel` with `local_fields=True`.

The model uses $E(s)=-\tfrac12s^TJs-h^Ts$ and big-endian spin labels. The circuits use positive Pauli coefficients and little-endian qubits. `circuit_coefficients` negates the coefficients and reverses the spin order, so state index x has the same energy in both conventions. All reference energies come from the real model's `.energies` property, with `rescaled=False` and `a=np.inf` for Metropolis acceptance.

`apply_unitary` lists targets from their least-significant local bit. `get_unitary` refuses large dense matrices; proposal columns can instead use small statevectors.


In [2]:
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)


def kron(*matrices):
    return reduce(np.kron, matrices)


def sequence(*matrices):
    """Matrix product, from left to right as written."""
    return reduce(np.dot, matrices)


def ket(i, n=1):
    vector = np.zeros((2**n, 1), complex)
    vector[i, 0] = 1
    return vector


def bra(i, n=1):
    return ket(i, n).conj().T


def ketbra(i, n=1):
    return ket(i, n) @ bra(i, n)


def encode(value, wires):
    return sum(((int(value) >> bit) & 1) << wire for bit, wire in enumerate(wires))


def decode(value, wires):
    return sum(((int(value) >> wire) & 1) << bit for bit, wire in enumerate(wires))


def apply_unitary(U, targets, n):
    targets = list(targets)
    mask = sum(1 << q for q in targets)
    result = np.zeros((2**n, 2**n), complex)
    for column in range(2**n):
        local = decode(column, targets)
        for row in range(len(U)):
            result[(column & ~mask) | encode(row, targets), column] = U[row, local]
    return result


def get_unitary(operation, big_endian=False):
    if operation.num_qubits > MAX_UNITARY_QUBITS:
        raise ValueError(f"Dense unitary refused: {operation.num_qubits} qubits.")
    if big_endian:
        if isinstance(operation, Gate):
            qc = QuantumCircuit(operation.num_qubits)
            qc.append(operation, range(operation.num_qubits))
        else:
            qc = operation
        operation = qc.reverse_bits()
    return Operator(operation).data


def check(actual, expected):
    np.testing.assert_allclose(actual, expected, atol=ATOL, rtol=0)


def instance(n):
    return RandomSherringtonKirkpatrickModel(n, local_fields=True, seed=100 + n)


def circuit_coefficients(model):
    return -model.h[::-1].copy(), -model.J[::-1, ::-1].copy()


## Primitives

Controlled matrices act on the lowest control qubits. Test angles include 0 and full turns, which detect incorrect conditional phases. The Givens matrices are also checked at zero and one-sided weights.


In [3]:
def rotation(pauli, theta):
    return np.cos(theta / 2) * np.eye(len(pauli)) - 1j * np.sin(theta / 2) * pauli


def controlled(U, controls):
    return np.eye(len(U) * 2**controls) + kron(U - np.eye(len(U)), ketbra(2**controls - 1, controls))


def givens(left, right):
    angle = np.arctan2(np.sqrt(right), np.sqrt(left)) if left + right else 0
    c, s = np.cos(angle), np.sin(angle)
    return (ketbra(0, 2) + ketbra(3, 2) + c * (ketbra(1, 2) + ketbra(2, 2))
            + s * (ket(2, 2) @ bra(1, 2) - ket(1, 2) @ bra(2, 2)))


check(get_unitary(Ccx()), controlled(X, 2))
for theta in (-2 * np.pi, -.7, 0., .123, np.pi, 2 * np.pi, 4 * np.pi):
    for factory, pauli, controls in (
        (Rx, X, 0), (Ry, Y, 0), (Crx, X, 1), (Cry, Y, 1), (Crz, Z, 1),
        (Ccrx, X, 2), (Ccry, Y, 2), (Ccrz, Z, 2),
        (Rzz, kron(Z, Z), 0), (Crzz, kron(Z, Z), 1), (Ccrzz, kron(Z, Z), 2),
    ):
        check(get_unitary(factory(theta)), controlled(rotation(pauli, theta), controls))
for left, right in ((1, 3), (3, 1), (2, 2), (0, 1), (1, 0), (0, 0)):
    check(get_unitary(GivensRotation(left, right)), givens(left, right))
    check(get_unitary(ControlledGivensRotation(left, right)), controlled(givens(left, right), 1))
    
print("Primitive matrices and conditional phases passed.")

Primitive matrices and conditional phases passed.


## Proposals

For each actual circuit P, extract $Q=V^\dagger P^\dagger S_{AB}PV$, with $V=|0\rangle_{B,\mathrm{extra}}\otimes I_A$. This equals the proposal matrix for the supplied symmetric-amplitude preparations. Full prepared columns also check phases and clean work.

Model-based tests start at n=2: the supplied model's one-spin `int_to_spin` returns a scalar, which its energy matrix multiplication does not support.


In [4]:
def trotter_matrix(model, t, gamma, r):
    n = model.n
    H_z = np.diag(model.energies)
    H_x = -sum(apply_unitary(X, [i], n) for i in range(n))
    half_x = expm(-.5j * t * gamma * H_x / r)
    return np.linalg.matrix_power(sequence(half_x, expm(-1j * t * (1 - gamma) * H_z / r), half_x), r)


def proposal_block(gate, n):
    assert gate.layout["A"] == list(range(n)) and gate.layout["B"] == list(range(n, 2 * n))
    if gate.num_qubits <= MAX_UNITARY_QUBITS:
        zero_input = kron(ket(0, gate.num_qubits - n), np.eye(2**n))
        prepared = get_unitary(gate) @ zero_input
    else:
        assert gate.num_qubits <= MAX_STATEVECTOR_QUBITS
        prepared = np.column_stack([
            Statevector.from_int(x, 2**gate.num_qubits).evolve(gate.definition).data
            for x in range(2**n)
        ])
    indices = np.arange(2**gate.num_qubits)
    mask = 2**n - 1
    swapped = (indices & ~((1 << (2 * n)) - 1)) | ((indices & mask) << n) | ((indices >> n) & mask)
    block = prepared.conj().T @ prepared[swapped]
    return block, prepared


def expected_preparation(amplitudes):
    n = len(amplitudes).bit_length() - 1
    return sum(amplitudes[y, x] * kron(ket(y, n), ket(x, n)) @ bra(x, n)
               for x, y in product(range(2**n), repeat=2))


def check_proposal(gate, expected, prepared=None):
    block, actual = proposal_block(gate, gate.n)
    check(block, expected)
    check(block, block.conj().T)
    check(block.sum(axis=0), np.ones(2**gate.n))
    if prepared is not None:
        check(actual, prepared)
    return block


def randomized_reference(gate, model):
    n, k = gate.n, gate.k
    prepared = np.zeros((2**gate.num_qubits, 2**n), complex)
    probability = np.zeros((2**n, 2**n))
    times = gate.t[0] + (np.arange(2**k) + .5) * (gate.t[1] - gate.t[0]) / 2**k
    gammas = gate.gamma[0] + (np.arange(2**k) + .5) * (gate.gamma[1] - gate.gamma[0]) / 2**k
    for p, q in product(range(2**k), repeat=2):
        U = trotter_matrix(model, times[p], gammas[q], gate.trotter_steps)
        probability += abs(U)**2 / 2**(2 * k)
        label = encode(p, gate.layout["t"]) | encode(q, gate.layout["gamma"])
        for x, y in product(range(2**n), repeat=2):
            row = label | encode(x, gate.layout["A"]) | encode(y, gate.layout["B"])
            prepared[row, x] = U[y, x] / 2**k
    return probability, prepared


### Uniform proposal

Compare with the uniform transition at beta=0, including the prepared amplitudes.


In [5]:
for n in (2, 3, 4):
    model = instance(n)
    reference = transition.create_transition_matrix("uniform", model, 0., a=np.inf, rescaled=False)
    check_proposal(ProposalUniform(n), reference, expected_preparation(np.sqrt(reference)))
print("Uniform proposal passed.")


Uniform proposal passed.


### Local proposal

Compare the single-spin-flip block with `local1` at beta=0.


In [6]:
for n in (2, 3, 4):
    model = instance(n)
    reference = transition.create_transition_matrix("local1", model, 0., a=np.inf, rescaled=False)
    check_proposal(ProposalLocal(n), reference, expected_preparation(np.sqrt(reference)))
print("Local proposal passed.")


Local proposal passed.


### Hamiltonian-simulation proposal

Compare finite-step circuits with the matching Strang matrix product. At gamma=0 or gamma=1, Trotterization is exact, so also compare with the module's exact-evolution reference using the model's own h,J arrays.


In [7]:
for n, r in product((2, 3, 4), (1, 2, 4)):
    model = instance(n)
    h, J = circuit_coefficients(model)
    U = trotter_matrix(model, .713, .413, r)
    check_proposal(ProposalHamSim(n, h, J, .713, .413, r), abs(U)**2, expected_preparation(U))
for n, gamma in product((2, 3, 4), (0., 1.)):
    model = instance(n)
    h, J = circuit_coefficients(model)
    reference = transition.create_proposal_matrix_quantum_exact(model.h, model.J, gamma, .37)
    block = check_proposal(ProposalHamSim(n, h, J, .37, gamma, 1), reference)
    check(np.linalg.eigvalsh(block), np.linalg.eigvalsh(reference))
print("Hamiltonian-simulation proposal passed.")


Hamiltonian-simulation proposal passed.


### Randomized Hamiltonian-simulation proposal

Compare the coherent midpoint-grid columns and the averaged proposal block. Gamma=1 additionally gives an exact module reference. Statevector columns keep k=2 and n=3 tests small without forming a full dense unitary.


In [8]:
for n, k, r in ((2, 1, 1), (2, 1, 2), (2, 2, 1), (3, 1, 1)):
    model = instance(n)
    h, J = circuit_coefficients(model)
    gate = ProposalHamSimRandomized(n, h, J, (.37, 1.13), (.17, .71), r, k)
    probability, prepared = randomized_reference(gate, model)
    check_proposal(gate, probability, prepared)
for n in (2, 3):
    model = instance(n)
    h, J = circuit_coefficients(model)
    gate = ProposalHamSimRandomized(n, h, J, (.2, .7), (1., 1.), 1, 1)
    reference = sum(transition.create_proposal_matrix_quantum_exact(model.h, model.J, 1., t)
                    for t in (.325, .575)) / 2
    check_proposal(gate, reference)
print("Randomized Hamiltonian-simulation proposal passed.")


Randomized Hamiltonian-simulation proposal passed.


## Reversible arithmetic

The evaluator below accepts only X, CX, and exact CCX (including inverses), recursively expanding composite arithmetic gates. It stores an entire computational-basis state in one Python integer. Every expected result is compared as a full bit string, so changed inputs or uncleared work cause a failure. Primitive tests above verify that CCX has no unwanted phases; linearity then extends these permutation checks to coherent inputs.


In [9]:
def classical_operations(qc, wires=None):
    wires = list(range(qc.num_qubits)) if wires is None else wires
    assert qc.global_phase == 0
    for inst in qc.data:
        op = inst.operation
        targets = [wires[qc.find_bit(q).index] for q in inst.qubits]
        name = op.name
        while name.endswith("_dg"):
            name = name[:-3]
        if name in {"x", "cx", "ccx"}:
            yield tuple(targets)
        elif op.definition is not None:
            yield from classical_operations(op.definition, targets)
        else:
            raise ValueError(f"Not an X/CX/CCX circuit: {op.name}")


def run_classical(operations, value):
    value = int(value)
    for wires in operations:
        if all((value >> q) & 1 for q in wires[:-1]):
            value ^= 1 << wires[-1]
    return value


def parities(n, value):
    return [((value >> i) & 1) for i in range(n)] + [
        (((value >> i) ^ (value >> j)) & 1) for i in range(n) for j in range(i + 1, n)]


def quantized_coefficients(h, J, F):
    n = len(h)
    values = list(h) + [J[i, j] for i in range(n) for j in range(i + 1, n)]
    alpha = 2 * float(np.sum(np.abs(values)))
    unit = alpha * 2.0**(-F) if alpha else 1.
    return [int(np.rint(value / unit)) for value in values], unit


def model_from_circuit_coefficients(n, coefficients, unit=1.):
    h = unit * np.asarray(coefficients[:n], dtype=float)
    J = np.zeros((n, n))
    J[np.triu_indices(n, 1)] = unit * np.asarray(coefficients[n:], dtype=float)
    J += J.T
    return IsingModel(J=-J[::-1, ::-1].copy(), h=-h[::-1].copy())


In [10]:
# Exhaustive small additions, including nonzero XOR targets.
for W in range(1, 6):
    gate = ReversibleCarryLookaheadAdder(W)
    ops, layout = list(classical_operations(gate.definition)), gate.layout
    for a, b, target in product(range(2**W), range(2**W), (0, 2**W - 1)):
        initial = encode(a, layout["a"]) | encode(b, layout["b"]) | encode(target, layout["output"])
        expected = initial ^ encode((a + b) % 2**W, layout["output"])
        assert run_classical(ops, initial) == expected
for M, W in product((1, 2, 3, 4, 7, 12), (1, 2, 3, 4, 8)):
    gate = ReversibleWallaceAdder(M, W)
    ops = list(classical_operations(gate.definition))
    inverse = list(classical_operations(gate.inverse().definition))
    trials = product(range(2**W), repeat=M) if M <= 3 and W <= 3 else rng.integers(0, 2**W, (40, M))
    for values in trials:
        initial = sum(encode(value, gate.layout["inputs"][i * W:(i + 1) * W]) for i, value in enumerate(values))
        initial |= encode(2**W - 1, gate.layout["output"])
        expected = initial ^ encode(sum(map(int, values)) % 2**W, gate.layout["output"])
        actual = run_classical(ops, initial)
        assert actual == expected and run_classical(inverse, actual) == initial
print("Prefix and Wallace additions passed.")


Prefix and Wallace additions passed.


In [11]:
for n, W in product((1, 2, 3, 4), (2, 4, 6)):
    coefficients = list(map(int, rng.integers(-2**(W - 1) + 1, 2**(W - 1), n * (n + 1) // 2)))
    gate = ReversibleConditionalLoader(n, coefficients, W)
    ops = list(classical_operations(gate.definition))
    for x, y in product(range(2**n), repeat=2):
        initial = encode(x, gate.layout["x"]) | encode(y, gate.layout["y"])
        expected = initial
        for i, (q, px, py) in enumerate(zip(coefficients, parities(n, x), parities(n, y))):
            expected ^= encode(q * (px - py), gate.layout["terms"][i * W:(i + 1) * W])
        assert run_classical(ops, initial) == expected
for n, F in product((2, 3, 4), (4, 8, 12)):
    model = instance(n)
    h, J = circuit_coefficients(model)
    gate = ReversibleDeltaEnergy(n, h, J, F)
    coefficients, unit = quantized_coefficients(h, J, F)
    assert gate.coefficients == coefficients
    check(gate.energy_unit, 2 * unit)
    integer_model = model_from_circuit_coefficients(n, coefficients)
    integer_energies = integer_model.energies
    ops = list(classical_operations(gate.definition))
    for x, y in product(range(2**n), repeat=2):
        delta = int((integer_energies[y] - integer_energies[x]) / 2)
        assert abs(delta) < 2**(gate.W - 1)
        initial = encode(x, gate.layout["x"]) | encode(y, gate.layout["y"])
        assert run_classical(ops, initial) == initial | encode(delta, gate.layout["delta"])
for W in range(1, 9):
    positive, comparator = ReversiblePositivePart(W), ReversibleComparator(W)
    pos_ops, cmp_ops = list(classical_operations(positive.definition)), list(classical_operations(comparator.definition))
    for value in range(2**W):
        signed = value if value < 2**(W - 1) else value - 2**W
        target = 2**W - 1
        initial = value | encode(target, positive.layout["positive"])
        assert run_classical(pos_ops, initial) == initial ^ encode(max(signed, 0), positive.layout["positive"])
        for flag in (0, 1):
            initial = value | (flag << W)
            assert run_classical(cmp_ops, initial) == initial ^ ((signed < 0) << W)
for m in range(1, 10):
    gate = ReversibleZeroFlag(m)
    ops = list(classical_operations(gate.definition))
    for bits, flag in product(range(2**m), (0, 1)):
        initial = bits | (flag << m)
        assert run_classical(ops, initial) == initial ^ ((bits == 0) << m)
    fanout = ReversibleFanout(m)
    ops = list(classical_operations(fanout.definition))
    inverse = list(classical_operations(fanout.inverse().definition))
    for bit in (0, 1):
        output = run_classical(ops, bit)
        assert output == bit * (2**m - 1) and run_classical(inverse, output) == bit
zero = ReversibleDeltaEnergy(2, np.zeros(2), np.zeros((2, 2)), 4)
assert all(run_classical(list(classical_operations(zero.definition)), x | (y << 2)) == x | (y << 2)
           for x, y in product(range(4), repeat=2))
print("Loaders, signed energies, positive parts, comparators, and control trees passed.")


Loaders, signed energies, positive parts, comparators, and control trees passed.


## Acceptance rotations

Test only the small `RotationalAcceptance` unitary, not the complete Boltzmann coin. For each nonnegative word p, extract its coin-register block using kets and bras; it must equal the tensor product of the expected controlled RY rotations. Its all-zero coin amplitude is $e^{-\beta u p/2}$, where u is the energy unit. Together with the classically verified positive part, its squared amplitude must reproduce Metropolis acceptance for both signs of the energy difference. A real field-only IsingModel supplies energies with differences u(y-x); constant offsets cancel. At least two model spins avoid its one-spin scalar case.


In [12]:
for W, beta in product((1, 2, 3, 4, 5), (0., .7, 4.)):
    unit = .125
    gate = RotationalAcceptance(W, beta, unit)
    U, C = get_unitary(gate), gate.num_coins
    for p in range(2**(W - 1)):
        embed = kron(np.eye(2**C), ket(p, W))
        block = embed.conj().T @ U @ embed
        angles = [2 * np.arccos(np.exp(-beta * unit * 2**j / 2)) if j < W - 1 and (p >> j) & 1 else 0
                  for j in range(C)]
        check(block, kron(*(rotation(Y, theta) for theta in reversed(angles))))
        check(block[0, 0], np.exp(-beta * unit * p / 2))
    n_model = max(2, W - 1)
    acceptance_model = IsingModel(
        J=np.zeros((n_model, n_model)),
        h=unit * 2.**np.arange(n_model - 1, -1, -1) / 2,
    )
    A = transition.create_acceptance_matrix(acceptance_model, beta, a=np.inf, rescaled=False)
    for x, y in product(range(2**(W - 1)), repeat=2):
        p = max(y - x, 0)
        check(abs(U[p, p])**2, A[y, x])
print("Acceptance rotation blocks and Metropolis probabilities passed.")


Acceptance rotation blocks and Metropolis probabilities passed.


## Reflection and Accept Path

Work inputs are zero. Compare the complete output columns, so these tests check clean work as well as the logical action. In little-endian tensor order the reflection is $2(|0\rangle\langle0|_{B,\mathrm{coins},\mathrm{extra}}\otimes I_A)-I$; its global sign is checked exactly. The accept path swaps A,B only on the all-zero coin subspace.


In [13]:
for n, C in ((1, 1), (1, 2), (2, 1), (2, 2), (2, 3), (3, 1)):
    gate = AcceptPath(n, C)
    d = 2**(2 * n)
    swap = np.zeros((d, d))
    for x, y in product(range(2**n), repeat=2):
        swap[y | (x << n), x | (y << n)] = 1
    expected = kron(ketbra(0, C), swap) + kron(np.eye(2**C) - ketbra(0, C), np.eye(d))
    embed = kron(ket(0, gate.work), np.eye(2**(2 * n + C)))
    check(get_unitary(gate) @ embed, embed @ expected)
for n, C, E in ((1, 1, 0), (1, 2, 0), (2, 1, 0), (2, 2, 0), (3, 1, 0), (1, 1, 1), (2, 1, 1)):
    gate = Reflection(n, C, E)
    active = 2 * n + C + E
    expected = 2 * kron(ketbra(0, n + C + E), np.eye(2**n)) - np.eye(2**active)
    embed = kron(ket(0, gate.work), np.eye(2**active))
    check(get_unitary(gate) @ embed, embed @ expected)
print("Accept-path swaps, reflection phases, and clean work passed.")


Accept-path swaps, reflection phases, and clean work passed.
